In [1]:
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
from data import UncertainDataset

In [3]:
from pathlib import Path

plots = Path("plots_mean_distribution")

In [4]:
from matplotlib import pyplot as plt
import seaborn as sns

def plot_mean_distribution(mean: xr.Dataset, plot: Path, sample: slice = slice(None)):
    fig, ax = plt.subplots()
    
    months = sorted(set(pd.to_datetime(mean.valid_time).month.values))
    years = sorted(set(pd.to_datetime(mean.valid_time).year.values))
    
    colours = [
        "tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", 
        "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan", 
        "gold", "teal"
    ]

    for year in years:
        for month in months:
            colour = colours[(month - 1) % 12]  # Assign a unique color to each month
            mean_values = mean.sel(valid_time=(mean.valid_time.dt.year == year) & (mean.valid_time.dt.month == month)).values.flatten()[sample]
            
            sns.kdeplot(mean_values, ax=ax, fill=True, color=colour, alpha=0.3, label=f"{year}/{month:02d}")

    ax.set_title(f"Distribution of {mean.attrs.get('long_name', 'Mean')} by Month")
    ax.set_xlabel(f"{mean.attrs.get('long_name', 'Mean')} [{mean.attrs.get('units', '')}]")
    ax.set_ylabel("Density")
    ax.legend(loc="upper right")
    
    plot.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot)
    
    plt.close(fig)


In [5]:
from tqdm import tqdm

In [6]:
from data import ds_single

for key in tqdm(ds_single.mean):
        plot_mean_distribution(ds_single.mean[key], plots / "single" / f"mean_distribution_{ds_single.mean[key].attrs.get('GRIB_shortName')}.png", slice(0, None, 1000))

100%|██████████| 226/226 [00:00<00:00, 1771799.45it/s]


In [7]:
from data import ds_pressure

for key in tqdm(ds_pressure.mean):
    for pressure in ds_pressure.mean[key].pressure_level.values:
        plot_mean_distribution(
            ds_pressure.mean[key].sel(pressure_level=pressure),
            plots / "pressure" / f"mean_distribution_{ds_pressure.mean[key].attrs.get('GRIB_shortName')}-{pressure}hPa.png",
            slice(0, None, 1000),
        )

  6%|▋         | 1/16 [02:22<35:30, 142.00s/it]/etc/ecmwf/ssd/ssd1/jupyterhub/ecm5945-jupyterhub/tmpdirs/ecm5945.33654218/ipykernel_2416386/3574751476.py:21: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(mean_values, ax=ax, fill=True, color=colour, alpha=0.3, label=f"{year}/{month:02d}")
/etc/ecmwf/ssd/ssd1/jupyterhub/ecm5945-jupyterhub/tmpdirs/ecm5945.33654218/ipykernel_2416386/3574751476.py:21: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(mean_values, ax=ax, fill=True, color=colour, alpha=0.3, label=f"{year}/{month:02d}")
/etc/ecmwf/ssd/ssd1/jupyterhub/ecm5945-jupyterhub/tmpdirs/ecm5945.33654218/ipykernel_2416386/3574751476.py:21: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(mean_values, ax=ax, fill=True, color=colour, alpha=0.3, labe